# ArmorVault — PP-OCRv5 Mobile Arabic + Qwen2.5 1.5B

Offline architecture feasibility test using a synthetic Saudi ID: official PP-OCRv5 Mobile ONNX models read text and geometry, Qwen2.5-1.5B proposes field mappings, and deterministic validators reject invalid dates and fake MRZ. No Google Drive or private documents are used.

**Colab proves extraction quality, not Android speed or battery use.**

In [ ]:
%pip install -q "rapidocr>=3.9,<4" "onnxruntime>=1.20" "huggingface_hub>=0.32,<1" "pyyaml>=6" "transformers==4.51.3" "tokenizers==0.21.4" "bitsandbytes==0.49.2" "accelerate>=1.4" pillow


In [ ]:
import gc, json, re, time
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve
import torch, yaml
from huggingface_hub import hf_hub_download
from PIL import Image
from IPython.display import display

root = Path('/content/armorvault-paddle-qwen')
root.mkdir(parents=True, exist_ok=True)
image_path = root / 'saudi_id_demo.png'
urlretrieve('https://raw.githubusercontent.com/almawti/armorvault-ocr-vl-gpu-lab/main/saudi_id_demo.png', image_path)
display(Image.open(image_path))
print({'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'image': str(image_path)})


In [ ]:
from rapidocr import RapidOCR

det_model = hf_hub_download('PaddlePaddle/PP-OCRv5_mobile_det_onnx', 'inference.onnx')
rec_model = hf_hub_download('PaddlePaddle/arabic_PP-OCRv5_mobile_rec_onnx', 'inference.onnx')
rec_config = hf_hub_download('PaddlePaddle/arabic_PP-OCRv5_mobile_rec_onnx', 'inference.yml')
config = yaml.safe_load(Path(rec_config).read_text(encoding='utf-8'))
characters = config['PostProcess']['character_dict']
keys_path = root / 'ppocrv5_arabic_dict.txt'
keys_path.write_text('\n'.join(str(char) for char in characters), encoding='utf-8')

ocr = RapidOCR(params={
    'Det.model_path': det_model,
    'Rec.model_path': rec_model,
    'Rec.rec_keys_path': str(keys_path),
    'Rec.rec_img_shape': [3, 48, 320],
})
started = time.perf_counter()
ocr_result = ocr(str(image_path), use_cls=False)
ocr_seconds = time.perf_counter() - started
texts = list(ocr_result.txts or [])
scores = [float(value) for value in (ocr_result.scores or [])]
boxes = ocr_result.boxes.tolist() if ocr_result.boxes is not None else []
lines = [
    {'id': index, 'text': text, 'confidence': round(scores[index], 4), 'box': boxes[index]}
    for index, text in enumerate(texts)
]
ocr_payload = {'model': 'PP-OCRv5 Mobile detector + Arabic Mobile recognizer (ONNX)', 'seconds': round(ocr_seconds, 3), 'lines': lines}
(root / 'ocr.json').write_text(json.dumps(ocr_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(ocr_payload, ensure_ascii=False, indent=2))
del ocr, ocr_result
gc.collect()


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto', quantization_config=quantization, torch_dtype=torch.float16).eval()
qwen_init_seconds = time.perf_counter() - started

schema = {
    'documentType': 'one of: saudi_national_id, passport, residence_permit, driver_license, invoice, certificate, other',
    'documentNumber': 'string or null',
    'holderName': 'string or null',
    'birthDate': 'YYYY-MM-DD or null',
    'issueDate': 'YYYY-MM-DD or null',
    'expiryDate': 'YYYY-MM-DD or null',
    'mrzLines': 'array of strings; empty unless actual ICAO MRZ with < characters exists',
    'evidence': 'object mapping each non-null field to OCR line id array'
}
prompt = '''You map OCR lines from identity documents to fields. Return one JSON object only.
Rules:
- Every value must be supported by the supplied OCR lines; never invent or repair characters silently.
- Distinguish birth, issue, and expiry labels. If a requested date label is absent, return null.
- Arabic and Western digits may be normalized, but preserve names as OCR read them.
- MRZ exists only when visible lines contain < and match passport/ID MRZ structure.
- Use only the allowed documentType values.
- Include OCR line ids in evidence for every non-null scalar field.

Output schema:\n''' + json.dumps(schema, ensure_ascii=False) + '\n\nOCR lines:\n' + json.dumps(lines, ensure_ascii=False)
messages = [{'role': 'system', 'content': 'You are a conservative document field mapper. Output valid JSON only.'}, {'role': 'user', 'content': prompt}]
model_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(model_input, return_tensors='pt').to(model.device)
started = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=320, do_sample=False, temperature=None, top_p=None)
qwen_seconds = time.perf_counter() - started
answer = tokenizer.decode(generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', answer, flags=re.I | re.S)
start, end = cleaned.find('{'), cleaned.rfind('}')
if start < 0 or end <= start:
    raise RuntimeError('Qwen did not return a JSON object. Raw output:\n' + answer)
proposal = json.loads(cleaned[start:end + 1])
print(json.dumps({'qwenInitializationSeconds': round(qwen_init_seconds, 2), 'qwenInferenceSeconds': round(qwen_seconds, 2), 'proposal': proposal}, ensure_ascii=False, indent=2))


In [ ]:
allowed_types = {'saudi_national_id', 'passport', 'residence_permit', 'driver_license', 'invoice', 'certificate', 'other'}
rejections = []
validated = dict(proposal)
if validated.get('documentType') not in allowed_types:
    rejections.append({'field': 'documentType', 'value': validated.get('documentType'), 'reason': 'not in allowlist'})
    validated['documentType'] = 'other'

for field in ('birthDate', 'issueDate', 'expiryDate'):
    value = validated.get(field)
    if value is not None:
        try:
            datetime.strptime(value, '%Y-%m-%d')
        except (TypeError, ValueError):
            rejections.append({'field': field, 'value': value, 'reason': 'invalid Gregorian date'})
            validated[field] = None

mrz = validated.get('mrzLines') or []
if mrz and not all('<' in line and len(line.replace(' ', '')) in {30, 36, 44} for line in mrz):
    rejections.append({'field': 'mrzLines', 'value': mrz, 'reason': 'not an ICAO-shaped MRZ'})
    validated['mrzLines'] = []

expected = {
    'documentType': 'saudi_national_id',
    'documentNumber': '123456789',
    'holderNameArabic': 'نورة بنت عبدالرحمن أحمد',
    'holderNameEnglish': 'NOURAH, ABDULRAHMAN AHMAD',
    'birthDate': '1992-04-28',
    'issueDate': None,
    'expiryDate': '2032-01-19',
    'mrzLines': []
}
final_result = {
    'models': {'ocr': ocr_payload['model'], 'mapper': model_name + ' 4-bit'},
    'timings': {'ocrSeconds': round(ocr_seconds, 3), 'qwenInitializationSeconds': round(qwen_init_seconds, 2), 'qwenInferenceSeconds': round(qwen_seconds, 2)},
    'proposal': proposal,
    'validated': validated,
    'rejections': rejections,
    'expectedForManualComparison': expected
}
(root / 'result.json').write_text(json.dumps(final_result, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nFINAL RESULT\n')
print(json.dumps(final_result, ensure_ascii=False, indent=2))


The final JSON is printed above and also saved temporarily at `/content/armorvault-paddle-qwen/result.json`. No upload or Google Drive step is required.